In [2]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

In [3]:
campaigns_df = pd.read_csv('data/campaigns.csv')
customers_df = pd.read_csv('data/customers.csv')
events_df = pd.read_csv('data/events.csv')
products_df = pd.read_csv('data/products.csv')
transactions_df = pd.read_csv('data/transactions.csv')

df_dict = {
    'campaigns' : campaigns_df, 
    'customers' : customers_df, 
    'events' : events_df,
    'products' : products_df,
    'transactions' : transactions_df
}

In [4]:
for name, df in df_dict.items() : 
    print(f'========= {name} ==========')
    df.info()

========= campaigns ==========
<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   campaign_id      50 non-null     int64  
 1   channel          50 non-null     str    
 2   objective        50 non-null     str    
 3   start_date       50 non-null     str    
 4   end_date         50 non-null     str    
 5   target_segment   50 non-null     str    
 6   expected_uplift  50 non-null     float64
dtypes: float64(1), int64(1), str(5)
memory usage: 2.9 KB
========= customers ==========
<class 'pandas.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype
---  ------               --------------   -----
 0   customer_id          100000 non-null  int64
 1   signup_date          100000 non-null  str  
 2   country              100000 non-null  str  
 3   age                  100000

**Insight**
- Terdapat kolom date yang masih belum sesuai tipe datanya
- Terdapat data yang kosong pada tabel transactions  

In [5]:
for name,df in df_dict.items() : 
    print(f'============ {name} ==============')
    print(df.describe(include='all'))

============ campaigns ==============
        campaign_id      channel     objective  start_date    end_date  \
count      50.00000           50            50          50          50   
unique          NaN            5             4          49          48   
top             NaN  Paid Search  Reactivation  2022-07-25  2022-09-29   
freq            NaN           11            15           2           2   
mean       25.50000          NaN           NaN         NaN         NaN   
std        14.57738          NaN           NaN         NaN         NaN   
min         1.00000          NaN           NaN         NaN         NaN   
25%        13.25000          NaN           NaN         NaN         NaN   
50%        25.50000          NaN           NaN         NaN         NaN   
75%        37.75000          NaN           NaN         NaN         NaN   
max        50.00000          NaN           NaN         NaN         NaN   

       target_segment  expected_uplift  
count              50        50.

Insight : 
- Campaign yang dilakukan lebih banyak dengan tujuan reactivation, melalui saluran paid search, dan dengan target segment new customer
- Customer yang melakukan sign up, customer lebih banyak tier bronze, yang berasal dari saluran organic, dan rata rata usia 35
- Customer yang menggunakan website, banyak melihat dan customer menggunakan mobile, dengan rata rata sesi durasi 1.3s 
- Product yang dijual, kebanyakan electronic dan brand Brand_7, rata rata product yang dijual dengahan harga kurang lebih 72
- transaksi dengan rata rata total produk yang terjual 1 dan revenue yang didapatkan rata rata 90

In [6]:
for name, df in df_dict.items() : 
    print(f'====== {name} ======')
    for col in df.select_dtypes(include=['object', 'str', 'category']).columns : 
        print(f'====== {col} ======')
        print(df[col].unique())

====== campaigns ======
====== channel ======
<StringArray>
['Paid Search', 'Email', 'Display', 'Social', 'Affiliate']
Length: 5, dtype: str
====== objective ======
<StringArray>
['Cross-sell', 'Retention', 'Reactivation', 'Acquisition']
Length: 4, dtype: str
====== start_date ======
<StringArray>
['2021-10-25', '2021-10-24', '2023-10-08', '2022-07-25', '2022-07-09',
 '2021-10-20', '2023-07-02', '2021-08-23', '2023-10-09', '2022-09-01',
 '2022-11-02', '2021-01-20', '2022-03-19', '2022-09-20', '2022-04-29',
 '2023-08-10', '2023-04-05', '2022-10-09', '2021-11-28', '2021-07-26',
 '2023-02-21', '2022-07-02', '2022-10-25', '2021-11-07', '2021-03-19',
 '2023-08-13', '2022-04-01', '2021-09-02', '2021-12-26', '2023-11-04',
 '2023-10-26', '2021-08-05', '2022-02-15', '2023-03-19', '2022-11-06',
 '2023-04-03', '2022-10-20', '2021-12-18', '2023-03-08', '2023-04-09',
 '2023-04-22', '2023-03-12', '2021-01-31', '2022-04-08', '2023-03-23',
 '2022-04-12', '2022-01-05', '2021-04-09', '2021-02-14']
Lengt

**Insight :** 
- Terdapat data yang bias dimana nilainya sama tetapi berbeda letter case pada kolom traffic_source di tabel events
- Terdapat letter case dan tanda '_' akan dibersihkan agar lebih rapi ketika report/visualisasi

## Data Cleaning 
- ganti tipe data tanggal dari str menjadi tipe datetime 
- mengubah data menjadi konsisten (mengubah huruf menjadi proper case, menghilangkan tanda _,angka di belakang koma, sorting datetime data)
- handling missing values

In [7]:
#rounding
campaigns_df['expected_uplift'] = campaigns_df['expected_uplift'].round(2)

In [8]:
#changing data to title case
events_df['traffic_source'] = events_df['traffic_source'].str.title()
events_df['device_type'] = events_df['device_type'].str.title()
events_df['event_type'] = events_df['event_type'].str.replace('_', ' ').str.title()
events_df['experiment_group'] = events_df['experiment_group'].str.replace('_', ' ').str.title()

In [9]:
unknown_data = pd.DataFrame({
    'product_id' : [0], 
    'category' : ['Unknown'], 
    'brand' : ['Unknown'], 
    'base_price' : ['Unknown'], 
    'launch_date' : [np.nan],
    'is_premium': ['Unknown']
})

products_df = pd.concat([products_df, unknown_data], 
          ignore_index=True)


In [10]:
#handling missing values
transactions_df['product_id'] = transactions_df['product_id'].fillna(0).astype(int)
transactions_df['gross_revenue'] = transactions_df['gross_revenue'].fillna(0)

In [11]:
events_df['product_id'] = events_df['product_id'].fillna(0).astype(int)

In [12]:
events_df.iloc[events_df['page_category'] == 'Bounce']

,event_id,timestamp,customer_id,session_id,event_type,product_id,device_type,traffic_source,campaign_id,page_category,session_duration_sec,experiment_group


In [13]:
events_df.isnull().sum()

event_id                    0
timestamp                   0
customer_id                 0
session_id                  0
event_type                  0
product_id                  0
device_type             40300
traffic_source              0
campaign_id                 0
page_category               0
session_duration_sec        0
experiment_group            0
dtype: int64

In [14]:
products_df['launch_date']

0       2021-08-02
1       2021-09-14
2       2021-01-18
3       2023-03-03
4       2022-04-19
           ...    
1996    2022-06-19
1997    2021-10-21
1998    2023-08-11
1999    2021-01-15
2000           NaN
Name: launch_date, Length: 2001, dtype: object

In [15]:
#changing to datetime type 
campaigns_df['start_date'] = pd.to_datetime(campaigns_df['start_date'])
campaigns_df['end_date'] = pd.to_datetime(campaigns_df['end_date'])
events_df['timestamp'] = pd.to_datetime(events_df['timestamp'])
products_df['launch_date'] = pd.to_datetime(products_df['launch_date'])
customers_df['signup_date'] = pd.to_datetime(customers_df['signup_date'])
transactions_df['timestamp'] = pd.to_datetime(transactions_df['timestamp'])

In [16]:
events_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000000 entries, 0 to 1999999
Data columns (total 12 columns):
 #   Column                Dtype         
---  ------                -----         
 0   event_id              int64         
 1   timestamp             datetime64[us]
 2   customer_id           int64         
 3   session_id            int64         
 4   event_type            str           
 5   product_id            int64         
 6   device_type           str           
 7   traffic_source        str           
 8   campaign_id           int64         
 9   page_category         str           
 10  session_duration_sec  float64       
 11  experiment_group      str           
dtypes: datetime64[us](1), float64(1), int64(5), str(5)
memory usage: 183.1 MB


In [17]:
events_df.iloc[events_df['page_category'] == 'bounce']

,event_id,timestamp,customer_id,session_id,event_type,product_id,device_type,traffic_source,campaign_id,page_category,session_duration_sec,experiment_group


In [18]:
#total missing values 
transactions_df.isnull().sum()

transaction_id      0
timestamp           0
customer_id         0
product_id          0
quantity            0
discount_applied    0
gross_revenue       0
campaign_id         0
refund_flag         0
dtype: int64

In [19]:
transactions_df.iloc[transactions_df['product_id'].isna()]

,transaction_id,timestamp,customer_id,product_id,quantity,discount_applied,gross_revenue,campaign_id,refund_flag


In [20]:
events_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2000000 entries, 0 to 1999999
Data columns (total 12 columns):
 #   Column                Dtype         
---  ------                -----         
 0   event_id              int64         
 1   timestamp             datetime64[us]
 2   customer_id           int64         
 3   session_id            int64         
 4   event_type            str           
 5   product_id            int64         
 6   device_type           str           
 7   traffic_source        str           
 8   campaign_id           int64         
 9   page_category         str           
 10  session_duration_sec  float64       
 11  experiment_group      str           
dtypes: datetime64[us](1), float64(1), int64(5), str(5)
memory usage: 183.1 MB


In [21]:
df_dict = {
    'dim_campaigns' : campaigns_df, 
    'dim_customers' : customers_df, 
    'fact_events' : events_df,
    'dim_products' : products_df,
    'fact_transactions' : transactions_df
}

### Integrate to Database

In [22]:
#connecting local database
server = 'LAPTOP-QOFMN6O5\SQLEXPRESS'
database = 'e_commerce_customer_and_marketing_analytics'
driver = 'ODBC Driver 17 for SQL Server'


connection_url = URL.create(
    "mssql+pyodbc", 
    query={
        "odbc_connect" : (
            f"DRIVER={{{driver}}};"
            f"SERVER={server};"
            f"DATABASE={database};"
            "Trusted_Connection=yes;"
        )
    }
)

#create engine for connection
engine = create_engine(connection_url)

In [23]:
with engine.connect() as conn : 
    print('tes')

tes


In [24]:
#load dataset to local database
for table_name, df in df_dict.items() : 
    df.to_sql(
        name= table_name, 
        con= engine,
        schema='dbo', 
        if_exists = 'replace', 
        index=False,
    )

In [25]:
customers_df

,customer_id,signup_date,country,age,gender,loyalty_tier,acquisition_channel
0,1,2021-04-08,BR,48,Male,Bronze,Referral
1,2,2023-04-28,IN,36,Female,Silver,Organic
2,3,2022-12-18,UK,35,Female,Silver,Organic
3,4,2022-04-26,US,45,Male,Silver,Paid Search
4,5,2022-04-20,IN,53,Male,Silver,Organic
...,...,...,...,...,...,...,...
99995,99996,2023-12-16,US,46,Male,Bronze,Paid Search
99996,99997,2021-01-20,IN,37,Male,Bronze,Organic
99997,99998,2022-09-22,US,38,Female,Bronze,Social
99998,99999,2021-01-06,IN,19,Female,Silver,Email


# TO DO
- menyamakan tipe data product_id di fact tabel dengan dim tabel agar dapat dijoin
- pass data cleaning ke database 
- analisis customer sql

In [26]:
#